# AntiSD on Gemma 4: five-minute demo

Companion to the blog post *Teaching Gemma 4 to Hesitate*. This notebook does **no training**.
It loads `google/gemma-4-E2B-it` plus the LoRA adapters trained in the post, then on a GSM8K
problem of your choice it

1. shows the results table from the published evaluation files (no compute),
2. renders the per-token PMI heatmap for the base model and the AntiSD adapter,
3. prints the two greedy traces side by side.

A free T4 is enough (the model loads in 4-bit there). Full reproduction, about seven hours on an
A100, is the other notebook: `antisd_gemma4_colab.ipynb`.

In [ ]:
# 1. Setup (about 2 minutes)
!pip install -q -U transformers peft datasets accelerate bitsandbytes
!pip uninstall -q -y torchao 2>/dev/null || true
import os, torch
if not os.path.exists("antisd-gemma4"):
    !git clone -q https://github.com/ranbir/antisd-gemma4.git
%cd antisd-gemma4
FOURBIT = "--load_in_4bit" if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory < 20e9) else ""
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| 4-bit:", bool(FOURBIT))

In [ ]:
# 2. Published results (read from the repo; nothing runs)
import json, glob, pandas as pd
RUN = "run2"
rows = []
for name, path in [("Gemma 4 E2B (base)", f"results/{RUN}/eval_base.json"),
                   ("+ GRPO", f"results/{RUN}/eval_grpo.json"),
                   ("+ AntiSD", f"results/{RUN}/eval_antisd.json")]:
    if os.path.exists(path):
        s = json.load(open(path))["summary"]
        rows.append({"method": name, "pass@1 (%)": round(100*s["accuracy"],1), "thought tokens": round(s["avg_thought_tokens"]),
                     "deliberation markers / trace": round(s["avg_deliberation_markers"],2),
                     "unfinished (%)": round(100*s["frac_unfinished"],1), "n": s["n_problems"]})
display(pd.DataFrame(rows).set_index("method")) if rows else print("results/ not published yet")

In [ ]:
# 3. Pick a held-out problem and the adapter to compare against the base model
IDX = 3                                            # GSM8K test index; try 21 or 28 for problems the base gets wrong
ADAPTER = "ribnar/gemma-4-e2b-it-antisd-run2"      # Hub id of the AntiSD adapter from the post (swap in your own)
!python inspect_pmi.py --gsm8k_index {IDX} --greedy --max_new_tokens 2048 {FOURBIT} \
    --html_out demo_base.html --json_out demo_base.json
!python inspect_pmi.py --gsm8k_index {IDX} --greedy --max_new_tokens 2048 {FOURBIT} --adapter_dir {ADAPTER} \
    --html_out demo_antisd.html --json_out demo_antisd.json

In [ ]:
# 4. Heatmaps: blue = tokens the answer-key teacher dislikes (rewarded by AntiSD)
from IPython.display import HTML, display
display(HTML(open("demo_base.html").read()))
display(HTML(open("demo_antisd.html").read()))

In [ ]:
# 5. Traces side by side
b, a = json.load(open("demo_base.json")), json.load(open("demo_antisd.json"))
print("PROBLEM:", b["problem"], "\nGOLD:", b["gold"].split("####")[-1].strip())
print("\n===== BASE =====\n", b["completion"])
print("\n===== ANTISD =====\n", a["completion"])
print("\nstats base:", b["stats"], "\nstats antisd:", a["stats"])